<a href="https://colab.research.google.com/github/AgrisJ/master_ledger/blob/main/Master_Ledger_%26_Web_UI_(V2_15).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install paho-mqtt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 3.1 MB/s eta 0:00:00


In [3]:
from google.colab import output

# This will provide a link to access the Flask app running on port 5000
output.serve_kernel_port_as_window(5000)
print("Click the link above once you start the Flask app in the next cell.")

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

Click the link above once you start the Flask app in the next cell.


In [ ]:
import json
import os
import threading
import time
import requests
from datetime import datetime
from flask import Flask, request, jsonify, render_template_string
import paho.mqtt.client as mqtt

# ==========================================
# CONFIGURATION & SETUP
# ==========================================
LEDGER_FILE = "ledger.json"
MQTT_BROKER = "127.0.0.1"
MQTT_PORT = 1883

app = Flask(__name__)

DEFAULT_LEDGER = {
    "system_rules": {
        "overtime_multiplier": 5.0
    },
    "users": {
        "user-01": {
            "name": "User 1",
            "allowance_schedule": {
                "Monday": 0, "Tuesday": 0, "Wednesday": 0, "Thursday": 0,
                "Friday": 0, "Saturday": 0, "Sunday": 0
            },
            "time_remaining_minutes": 0,
            "bonus_time_minutes": 0,
            "enforce_daily_hard_limit": True,
            "last_reset_date": "",
            "active_station": None
        }
    },
    "tokens": {},
    "stations": {}
}

# ==========================================
# DATA MANAGEMENT
# ==========================================
def normalize_mac(mac):
    """Standardizes MAC addresses to uppercase with colons to prevent duplicates."""
    clean = mac.replace(":", "").upper()
    if len(clean) == 12:
        return ":".join(clean[i:i+2] for i in range(0, len(clean), 2))
    return mac

def load_ledger():
    if not os.path.exists(LEDGER_FILE):
        save_ledger(DEFAULT_LEDGER)
        return DEFAULT_LEDGER

    with open(LEDGER_FILE, "r") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            data = DEFAULT_LEDGER

        if "stations" not in data: data["stations"] = DEFAULT_LEDGER["stations"]
        if "tokens" not in data: data["tokens"] = DEFAULT_LEDGER["tokens"]
        if "system_rules" not in data: data["system_rules"] = DEFAULT_LEDGER["system_rules"]

        updated = False

        # Data Migration: Fix MAC Address duplicates
        cleaned_tokens = {}
        for mac, uid in data["tokens"].items():
            norm_mac = normalize_mac(mac)
            if norm_mac != mac:
                updated = True

            # Prioritize assigned tokens over unassigned duplicates
            if norm_mac in cleaned_tokens and cleaned_tokens[norm_mac] is not None:
                if uid is None: continue
            cleaned_tokens[norm_mac] = uid

        if data["tokens"] != cleaned_tokens:
            data["tokens"] = cleaned_tokens
            updated = True

        if updated: save_ledger(data)
        return data

def save_ledger(data):
    with open(LEDGER_FILE, "w") as f:
        json.dump(data, f, indent=4)

def check_daily_reset(ledger):
    today = datetime.now().strftime("%Y-%m-%d")
    day_of_week = datetime.now().strftime("%A")
    updated = False

    for user_id, data in ledger["users"].items():
        if data.get("last_reset_date") != today:
            base_allowance = data["allowance_schedule"].get(day_of_week, 0)

            # --- PENALTY CALCULATION ---
            carryover_debt = 0
            if data.get("time_remaining_minutes", 0) < 0:
                multiplier = float(ledger.get("system_rules", {}).get("overtime_multiplier", 1.0))
                overtime = abs(data["time_remaining_minutes"])
                carryover_debt = int(overtime * multiplier)
                print(f"[RESET] Applying {carryover_debt}m penalty to {user_id} for {overtime}m overtime.")

            # Apply new balance (can instantly drop them into the negative if debt > allowance!)
            data["time_remaining_minutes"] = base_allowance - carryover_debt
            data["bonus_time_minutes"] = 0
            data["last_reset_date"] = today
            data["active_station"] = None
            updated = True

    if updated: save_ledger(ledger)
    return ledger

# ==========================================
# WEBHOOK DISPATCHER
# ==========================================
def _fire_webhooks(urls, method, action, station_name, delay):
    """Background worker to fire webhooks sequentially without blocking the MQTT thread."""
    for i, url in enumerate(urls):
        try:
            if method == "POST":
                requests.post(url, timeout=3)
            else:
                requests.get(url, timeout=3)
            print(f"[DISPATCHER] Fired {action} webhook for {station_name} to: {url[:30]}...")
        except Exception as e:
            print(f"[DISPATCHER] Webhook failed for {station_name} at {url[:30]}... : {e}")

        # If there is another URL in the list, and a delay is configured, sleep
        if i < len(urls) - 1 and delay > 0:
            print(f"[DISPATCHER] Waiting {delay}s before next webhook for {station_name}...")
            time.sleep(delay)

def dispatch_webhook(station_id, action, force=False):
    ledger = load_ledger()
    station = ledger["stations"].get(station_id)

    if not station or station["type"] != "webhook":
        return

    # Prevent normal token removals from locking a station if Admin Override is ON
    if action == "LOCK" and station.get("admin_override", False) and not force:
        print(f"[DISPATCHER] Admin Override ON for {station['name']}. Ignoring lock request.")
        return

    # Get the URL string and split by commas
    url_string = station["integration"].get("unlock_url", "") if action == "UNLOCK" else station["integration"].get("lock_url", "")
    method = station["integration"].get("method", "POST")
    delay = float(station["integration"].get("delay_seconds", 0.0))

    urls = [u.strip() for u in url_string.split(',') if u.strip()]
    if not urls: return

    # Spin off a background thread so time.sleep() doesn't freeze the Pi's MQTT listener
    threading.Thread(target=_fire_webhooks, args=(urls, method, action, station["name"], delay), daemon=True).start()

# ==========================================
# MQTT BACKGROUND THREAD
# ==========================================
mqtt_client = mqtt.Client(client_id="PiMasterLedger")

def on_connect(client, userdata, flags, rc):
    print(f"[MQTT] Connected! Ready for Tokens.")
    client.subscribe("screentime/tokens/+/session")
    client.subscribe("screentime/tokens/hello")
    client.subscribe("screentime/scans")

def on_message(client, userdata, msg):
    try:
        topic = msg.topic
        payload = json.loads(msg.payload.decode())
        ledger = load_ledger()

        if topic == "screentime/tokens/hello":
            mac = normalize_mac(payload.get("mac"))
            if mac and mac not in ledger["tokens"]:
                ledger["tokens"][mac] = None
                save_ledger(ledger)

        elif "session" in topic:
            mac = normalize_mac(topic.split("/")[2])
            user_id = ledger["tokens"].get(mac)
            if user_id and user_id in ledger["users"] and payload.get("status") == "idle":
                ledger["users"][user_id]["time_remaining_minutes"] -= payload.get("consumed_minutes", 0)
                save_ledger(ledger)
                broadcast_balance(user_id, ledger["users"][user_id])

        elif topic == "screentime/scans":
            mac = normalize_mac(payload.get("mac"))
            tag = payload.get("tag")
            action = payload.get("action")

            user_id = ledger["tokens"].get(mac)
            if user_id and tag in ledger["stations"]:
                # Track the active station for the user
                ledger["users"][user_id]["active_station"] = tag if action == "START" else None
                save_ledger(ledger)

                # If they have time, OR they are stopping a session, fire webhook
                if ledger["users"][user_id]["time_remaining_minutes"] > 0 or action == "STOP":
                    dispatch_webhook(tag, "UNLOCK" if action == "START" else "LOCK")

                broadcast_balance(user_id, ledger["users"][user_id])

    except Exception as e:
        print(f"[MQTT] Error processing message: {e}")

def broadcast_balance(user_id, user_data):
    ledger = load_ledger()

    # Check if they are on a Manual station. If so, DO NOT enforce the limit on the token!
    is_enforced = user_data.get("enforce_daily_hard_limit", True)
    active_station_id = user_data.get("active_station")

    if active_station_id and active_station_id in ledger["stations"]:
        if ledger["stations"][active_station_id]["type"] == "manual":
            is_enforced = False

    payload = {
        "time_remaining_minutes": user_data["time_remaining_minutes"],
        "bonus_time_minutes": user_data["bonus_time_minutes"],
        "enforce_limit": is_enforced
    }

    for mac, assigned_user in ledger["tokens"].items():
        if assigned_user == user_id:
            topic = f"screentime/tokens/{mac}/balance"
            mqtt_client.publish(topic, json.dumps(payload), retain=True)

def start_mqtt():
    mqtt_client.on_connect = on_connect
    mqtt_client.on_message = on_message
    try:
        mqtt_client.connect(MQTT_BROKER, MQTT_PORT, 60)
        mqtt_client.loop_forever()
    except Exception as e:
        print(f"[MQTT] Failed to start: {e}")

# ==========================================
# WEB API & ROUTES
# ==========================================
@app.route("/")
def index():
    return render_template_string(HTML_TEMPLATE)

@app.route("/api/ledger", methods=["GET"])
def get_ledger():
    return jsonify(check_daily_reset(load_ledger()))

@app.route("/api/system", methods=["POST"])
def update_system():
    ledger = load_ledger()
    ledger["system_rules"]["overtime_multiplier"] = float(request.json.get("multiplier", 1.0))
    save_ledger(ledger)
    return jsonify({"status": "success"})

@app.route("/api/topup/<user_id>", methods=["POST"])
def topup(user_id):
    minutes = request.json.get("minutes", 0)
    ledger = load_ledger()
    if user_id in ledger["users"]:
        ledger["users"][user_id]["bonus_time_minutes"] += minutes
        ledger["users"][user_id]["time_remaining_minutes"] += minutes
        save_ledger(ledger)
        broadcast_balance(user_id, ledger["users"][user_id])
        return jsonify({"status": "success"})
    return jsonify({"status": "error"}), 404

@app.route("/api/schedule/<user_id>", methods=["POST"])
def update_schedule(user_id):
    schedule = request.json.get("schedule", {})
    ledger = load_ledger()
    if user_id in ledger["users"]:
        day_of_week = datetime.now().strftime("%A")
        old_today_mins = ledger["users"][user_id]["allowance_schedule"].get(day_of_week, 0)
        new_today_mins = schedule.get(day_of_week, 0)
        difference = new_today_mins - old_today_mins

        ledger["users"][user_id]["allowance_schedule"] = schedule
        ledger["users"][user_id]["time_remaining_minutes"] += difference

        save_ledger(ledger)
        broadcast_balance(user_id, ledger["users"][user_id])
        return jsonify({"status": "success"})
    return jsonify({"status": "error"}), 404

@app.route("/api/station", methods=["POST"])
def save_station():
    data = request.json
    tag_id = data.get("tag_id")
    ledger = load_ledger()

    if tag_id:
        ledger["stations"][tag_id] = {
            "name": data.get("name", "New Station"),
            "type": data.get("type", "webhook"),
            "admin_override": False,
            "integration": {
                "unlock_url": data.get("unlock_url", ""),
                "lock_url": data.get("lock_url", ""),
                "method": data.get("method", "POST"),
                "delay_seconds": float(data.get("delay_seconds", 0.0))
            }
        }
        save_ledger(ledger)
        return jsonify({"status": "success"})
    return jsonify({"status": "error"}), 400

@app.route("/api/station/<tag_id>", methods=["DELETE"])
def delete_station(tag_id):
    ledger = load_ledger()
    if tag_id in ledger["stations"]:
        del ledger["stations"][tag_id]
        save_ledger(ledger)
        return jsonify({"status": "success"})
    return jsonify({"status": "error"}), 404

@app.route("/api/station/<tag_id>/override", methods=["POST"])
def toggle_station_override(tag_id):
    override_state = request.json.get("override", False)
    ledger = load_ledger()
    if tag_id in ledger["stations"]:
        ledger["stations"][tag_id]["admin_override"] = override_state
        save_ledger(ledger)

        # Fire webhook immediately
        action = "UNLOCK" if override_state else "LOCK"
        dispatch_webhook(tag_id, action, force=True)
        return jsonify({"status": "success"})
    return jsonify({"status": "error"}), 404

@app.route("/api/token/<mac>", methods=["POST"])
def assign_token(mac):
    user_id = request.json.get("user_id")
    ledger = load_ledger()
    norm_mac = normalize_mac(mac)
    if norm_mac in ledger["tokens"]:
        ledger["tokens"][norm_mac] = user_id if user_id else None
        save_ledger(ledger)
        if user_id and user_id in ledger["users"]:
            broadcast_balance(user_id, ledger["users"][user_id])
        return jsonify({"status": "success"})
    return jsonify({"status": "error"}), 404

@app.route("/api/user/<user_id>", methods=["POST"])
def update_user(user_id):
    data = request.json
    ledger = load_ledger()
    if user_id in ledger["users"]:
        ledger["users"][user_id]["name"] = data.get("name", ledger["users"][user_id]["name"])
        save_ledger(ledger)
        return jsonify({"status": "success"})
    return jsonify({"status": "error"}), 404

@app.route("/api/user", methods=["PUT"])
def add_user():
    ledger = load_ledger()
    count = 1
    while f"user-{count:02d}" in ledger["users"]: count += 1
    new_id = f"user-{count:02d}"
    ledger["users"][new_id] = {
        "name": f"User {count}",
        "allowance_schedule": { day: 0 for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"] },
        "time_remaining_minutes": 0, "bonus_time_minutes": 0,
        "enforce_daily_hard_limit": True,
        "last_reset_date": datetime.now().strftime("%Y-%m-%d"),
        "active_station": None
    }
    save_ledger(ledger)
    return jsonify({"status": "success", "id": new_id})

# ==========================================
# FRONTEND HTML & JS
# ==========================================
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Family Screen Time</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <style>
        :root { --brand-orange: #E60012; --nintendo-orange: #FF7D00; --bg-light: #F2F2F2; --card-white: #FFFFFF; --text-dark: #333333;}
        body { background-color: var(--bg-light); color: var(--text-dark); font-family: 'Segoe UI', sans-serif; -webkit-tap-highlight-color: transparent;}
        .app-header { background-color: var(--nintendo-orange); color: white; }
        .kid-card { background: var(--card-white); border-radius: 24px; box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.05); transition: background-color 0.3s; }
        .btn-topup { background-color: #F8F9FA; border: 2px solid #E5E7EB; border-radius: 9999px; font-weight: bold; }
        .btn-topup:active { transform: scale(0.95); }
        .btn-topup.orange { border-color: var(--nintendo-orange); color: var(--nintendo-orange); }
        .time-display { font-size: 3.5rem; font-weight: 800; line-height: 1; color: var(--nintendo-orange); transition: color 0.3s;}
        .time-display.pulse { color: #E60012; }
        .input-day { background-color: #F3F4F6; border: none; border-radius: 12px; padding: 8px 12px; width: 80px; text-align: center; font-weight: 600; }
        .tab-btn { opacity: 0.6; transition: opacity 0.2s; padding-bottom: 4px; border-bottom: 2px solid transparent; }
        .tab-btn.active { opacity: 1; border-color: white; font-weight: bold; }
        .input-text { width: 100%; padding: 10px; border-radius: 8px; border: 1px solid #ccc; margin-bottom: 12px; }

        .battery-bar-container { width: 100%; height: 28px; background: #e5e7eb; border-radius: 14px; overflow: hidden; display: flex; }
        .battery-base { background: #9CA3AF; height: 100%; transition: width 0.5s ease-out; display:flex; align-items:center; justify-content:center; color:white; font-size:12px; font-weight:bold; }
        .battery-bonus { background: var(--nintendo-orange); height: 100%; transition: width 0.5s ease-out; display:flex; align-items:center; justify-content:center; color:white; font-size:12px; font-weight:bold;}
        .battery-overtime { background: linear-gradient(90deg, #f87171, #ef4444); height: 100%; transition: width 0.5s ease-out; width: 100%; display:flex; align-items:center; justify-content:center; color:white; font-size:12px; font-weight:bold;}
        .overtime-bg { background-color: #FEF2F2 !important; border: 2px solid #FECACA; }
        .text-overtime { color: #DC2626 !important; }
    </style>
</head>
<body class="pb-12">
    <header class="app-header p-6 rounded-b-3xl mb-8">
        <div class="max-w-4xl mx-auto flex justify-between items-end">
            <h1 class="text-3xl font-black tracking-tight">Parental Controls</h1>
            <div class="flex gap-4 text-lg">
                <button onclick="switchTab('dashboard')" id="tab-dashboard" class="tab-btn active">Dashboard</button>
                <button onclick="switchTab('setup')" id="tab-setup" class="tab-btn">System Setup</button>
            </div>
        </div>
    </header>

    <main class="max-w-4xl mx-auto px-4 grid grid-cols-1 md:grid-cols-2 gap-8" id="view-dashboard"></main>

    <main class="max-w-4xl mx-auto px-4 hidden space-y-8" id="view-setup">
        <div class="kid-card p-6">
            <h2 class="text-2xl font-black text-gray-800 mb-4">System Rules</h2>
            <div class="bg-red-50 border border-red-100 p-5 rounded-2xl">
                <h3 class="font-bold text-red-800 mb-2">Manual Station Overtime Penalty</h3>
                <p class="text-sm text-red-600 mb-4">When a child plays into negative time on a manual station, multiply their debt and deduct it from tomorrow's allowance.</p>
                <div class="flex items-center gap-3">
                    <span class="font-bold text-red-800">Multiplier:</span>
                    <input type="number" id="penalty-input" class="input-text !mb-0 w-24 text-center font-bold text-lg" step="0.5">
                    <span class="font-bold text-red-800 flex-1">x</span>
                    <button onclick="saveSystemRules()" id="save-rules-btn" class="bg-red-600 text-white font-bold py-2 px-6 rounded-xl">Save</button>
                </div>
            </div>
        </div>

        <div class="kid-card p-6">
            <div class="flex justify-between items-center mb-4">
                <h2 class="text-2xl font-black text-gray-800">User Management</h2>
                <button onclick="addUser()" class="bg-gray-800 text-white font-bold py-2 px-4 rounded-lg text-sm">+ Add User</button>
            </div>
            <div id="users-container" class="space-y-4"></div>
        </div>

        <div class="kid-card p-6">
            <div class="flex justify-between items-center mb-4">
                <h2 class="text-2xl font-black text-gray-800">Hardware Tokens</h2>
            </div>
            <div id="tokens-container" class="space-y-4"></div>
        </div>

        <div class="kid-card p-6">
            <div class="flex justify-between items-center mb-4">
                <h2 class="text-2xl font-black text-gray-800">Station Integrations</h2>
            </div>
            <div id="stations-container" class="space-y-6 mb-6"></div>

            <div class="bg-gray-50 p-5 rounded-2xl border border-gray-200">
                <h3 class="font-bold text-gray-700 mb-3">Add / Edit Station</h3>
                <input type="text" id="new-station-tag" placeholder="NFC Tag ID (e.g. 04CE...)" class="input-text font-mono">
                <input type="text" id="new-station-name" placeholder="Station Name (e.g. Zorin PC)" class="input-text">
                <select id="new-station-type" class="input-text bg-white">
                    <option value="webhook">Webhook (Smart Home/PC)</option>
                    <option value="manual">Manual (Nintendo Switch)</option>
                </select>
                <input type="text" id="new-station-unlock" placeholder="Unlock Webhook URL(s) - Comma-separated" class="input-text">
                <input type="text" id="new-station-lock" placeholder="Lock Webhook URL(s) - Comma-separated" class="input-text">

                <div class="flex items-center gap-3 mb-4 mt-2">
                    <span class="text-sm font-bold text-gray-600">Delay between multiple webhooks:</span>
                    <input type="number" id="new-station-delay" placeholder="0" class="input-text !mb-0 w-24 text-center font-bold" step="0.5" min="0">
                    <span class="text-sm text-gray-500">sec</span>
                </div>

                <button onclick="saveStation()" class="w-full bg-gray-800 text-white font-bold py-3 rounded-xl">Save Station</button>
            </div>
        </div>
    </main>

    <script>
        const daysOfWeek = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"];
        let currentLedger = {};
        let currentTab = 'dashboard';

        function switchTab(tabId) {
            currentTab = tabId;
            document.getElementById('view-dashboard').classList.toggle('hidden', tabId !== 'dashboard');
            document.getElementById('view-setup').classList.toggle('hidden', tabId !== 'setup');
            document.getElementById('tab-dashboard').classList.toggle('active', tabId === 'dashboard');
            document.getElementById('tab-setup').classList.toggle('active', tabId === 'setup');
            if (tabId === 'setup') renderSetup(currentLedger);
            if (tabId === 'dashboard') renderDashboard(currentLedger);
        }

        async function fetchLedger() {
            try {
                const res = await fetch('/api/ledger');
                currentLedger = await res.json();

                // Pause rendering if user is currently typing in any input field
                const activeEl = document.activeElement;
                const isTyping = activeEl && (activeEl.tagName === 'INPUT' || activeEl.tagName === 'SELECT');

                if (!isTyping) {
                    if (currentTab === 'dashboard') renderDashboard(currentLedger);
                    if (currentTab === 'setup') renderSetup(currentLedger);
                }
            } catch (err) {
                console.error("Ledger Sync Failed", err);
            }
        }

        async function saveSystemRules() {
            const mult = document.getElementById('penalty-input').value;
            const btn = document.getElementById('save-rules-btn');
            btn.innerText = "Saved!"; btn.classList.add("bg-green-500");
            await fetch('/api/system', { method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify({ multiplier: mult }) });
            document.activeElement.blur();
            setTimeout(() => { btn.innerText = "Save"; btn.classList.remove("bg-green-500"); fetchLedger(); }, 1500);
        }

        async function topUp(userId, minutes) {
            await fetch(`/api/topup/${userId}`, { method: 'POST', headers: { 'Content-Type': 'application/json' }, body: JSON.stringify({ minutes }) });
            fetchLedger();
        }

        async function toggleOverride(tagId, currentState) {
            await fetch(`/api/station/${tagId}/override`, {
                method: 'POST', headers: { 'Content-Type': 'application/json' }, body: JSON.stringify({ override: !currentState })
            });
            fetchLedger();
            setTimeout(fetchLedger, 500); // quick refresh for UI
        }

        async function addUser() {
            await fetch('/api/user', { method: 'PUT' });
            fetchLedger();
            setTimeout(() => renderSetup(currentLedger), 500);
        }

        async function saveSchedule(userId) {
            const schedule = {};
            daysOfWeek.forEach(day => {
                schedule[day] = parseInt(document.getElementById(`${userId}-${day}`).value) || 0;
            });
            const btn = document.getElementById(`save-btn-${userId}`);
            btn.innerText = "Saved!"; btn.classList.add("bg-green-500");
            await fetch(`/api/schedule/${userId}`, { method: 'POST', headers: { 'Content-Type': 'application/json' }, body: JSON.stringify({ schedule }) });
            document.activeElement.blur();
            setTimeout(() => { btn.innerText = "Save Schedule"; btn.classList.remove("bg-green-500"); fetchLedger(); }, 1500);
        }

        async function assignToken(mac, userId) {
            await fetch(`/api/token/${mac}`, { method: 'POST', headers: { 'Content-Type': 'application/json' }, body: JSON.stringify({ user_id: userId }) });
        }

        async function saveUserName(userId) {
            const newName = document.getElementById(`name-input-${userId}`).value;
            await fetch(`/api/user/${userId}`, { method: 'POST', headers: { 'Content-Type': 'application/json' }, body: JSON.stringify({ name: newName }) });
            const btn = document.getElementById(`name-btn-${userId}`);
            btn.innerText = "Saved!"; btn.classList.add("bg-green-500", "text-white");
            document.activeElement.blur();
            setTimeout(() => { btn.innerText = "Rename"; btn.classList.remove("bg-green-500", "text-white"); fetchLedger(); }, 1500);
        }

        async function deleteStation(tagId) {
            const btn = document.getElementById(`del-btn-${tagId}`);
            if (btn.innerText === 'Delete') {
                btn.innerText = 'Sure?';
                btn.classList.add('!bg-red-500', '!text-white');
                setTimeout(() => {
                    if(document.getElementById(`del-btn-${tagId}`)) {
                        btn.innerText = 'Delete';
                        btn.classList.remove('!bg-red-500', '!text-white');
                    }
                }, 3000);
                return;
            }
            await fetch(`/api/station/${tagId}`, { method: 'DELETE' });
            fetchLedger();
            setTimeout(() => renderSetup(currentLedger), 500);
        }

        function editStation(tagId, b64Data) {
            const data = JSON.parse(atob(b64Data));
            document.getElementById('new-station-tag').value = tagId;
            document.getElementById('new-station-name').value = data.name;
            document.getElementById('new-station-type').value = data.type;
            document.getElementById('new-station-unlock').value = data.integration?.unlock_url || '';
            document.getElementById('new-station-lock').value = data.integration?.lock_url || '';
            document.getElementById('new-station-delay').value = data.integration?.delay_seconds || 0;
            document.getElementById('new-station-tag').scrollIntoView({ behavior: 'smooth', block: 'center' });
        }

        async function saveStation() {
            const payload = {
                tag_id: document.getElementById('new-station-tag').value,
                name: document.getElementById('new-station-name').value,
                type: document.getElementById('new-station-type').value,
                unlock_url: document.getElementById('new-station-unlock').value,
                lock_url: document.getElementById('new-station-lock').value,
                delay_seconds: parseFloat(document.getElementById('new-station-delay').value) || 0.0,
                method: "POST"
            };
            if (!payload.tag_id) return alert("Tag ID is required!");
            await fetch('/api/station', { method: 'POST', headers: { 'Content-Type': 'application/json' }, body: JSON.stringify(payload) });

            document.getElementById('new-station-tag').value = '';
            document.getElementById('new-station-name').value = '';
            document.getElementById('new-station-unlock').value = '';
            document.getElementById('new-station-lock').value = '';
            document.getElementById('new-station-delay').value = '';
            document.activeElement.blur();
            fetchLedger();
            setTimeout(() => renderSetup(currentLedger), 500);
        }

        function formatTime(minutes) {
            if (minutes <= 0) return "0h 0m";
            const h = Math.floor(minutes / 60);
            return `${h > 0 ? h + 'h ' : ''}${minutes % 60}m`;
        }

        function renderDashboard(ledger) {
            const container = document.getElementById('view-dashboard');
            let newHtml = '';

            for (const [userId, data] of Object.entries(ledger.users)) {
                const isOvertime = data.time_remaining_minutes < 0;
                const absTime = Math.abs(data.time_remaining_minutes);
                let barHtml = '';

                // Construct the active station badge HTML
                let activeStationHtml = '';
                if (data.active_station && ledger.stations[data.active_station]) {
                    const stationName = ledger.stations[data.active_station].name;
                    activeStationHtml = `
                        <div class="flex items-center gap-2 bg-green-50 px-3 py-1.5 rounded-full border border-green-200">
                            <span class="relative flex h-2.5 w-2.5">
                              <span class="animate-ping absolute inline-flex h-full w-full rounded-full bg-green-400 opacity-75"></span>
                              <span class="relative inline-flex rounded-full h-2.5 w-2.5 bg-green-500"></span>
                            </span>
                            <span class="text-xs font-bold text-green-700 uppercase tracking-wide">${stationName}</span>
                        </div>
                    `;
                } else {
                     activeStationHtml = `
                        <div class="flex items-center gap-2 bg-gray-50 px-3 py-1.5 rounded-full border border-gray-100">
                            <span class="relative inline-flex rounded-full h-2.5 w-2.5 bg-gray-300"></span>
                            <span class="text-xs font-bold text-gray-400 uppercase tracking-wide">Offline</span>
                        </div>
                    `;
                }

                if (isOvertime) {
                    const mult = ledger.system_rules?.overtime_multiplier || 1.0;
                    const penalty = Math.floor(absTime * mult);
                    barHtml = `
                        <div class="battery-bar-container bg-red-100 flex justify-end">
                            <div class="battery-overtime">Overtime Active</div>
                        </div>
                        <div class="text-center mt-2 text-sm font-bold text-red-500 tracking-wide">
                            Tomorrow's Penalty (x${mult}): -${penalty}m
                        </div>
                    `;
                } else {
                    const bonusLeft = Math.min(data.time_remaining_minutes, data.bonus_time_minutes);
                    const baseLeft = Math.max(0, data.time_remaining_minutes - data.bonus_time_minutes);
                    const total = data.time_remaining_minutes || 1;
                    const basePct = (baseLeft / total) * 100;
                    const bonusPct = (bonusLeft / total) * 100;
                    barHtml = `
                        <div class="battery-bar-container">
                            ${baseLeft > 0 ? `<div class="battery-base" style="width: ${basePct}%">${baseLeft}m</div>` : ''}
                            ${bonusLeft > 0 ? `<div class="battery-bonus" style="width: ${bonusPct}%">+${bonusLeft}m</div>` : ''}
                        </div>
                        <div class="flex justify-between mt-2 text-xs font-bold text-gray-400 px-1">
                            <span>Base Time</span><span>Bonus Time</span>
                        </div>
                    `;
                }

                let scheduleHtml = '';
                daysOfWeek.forEach(day => {
                    scheduleHtml += `
                        <div class="flex justify-between items-center py-2 border-b border-gray-100 last:border-0">
                            <span class="text-gray-600 font-medium">${day.slice(0,3)}</span>
                            <div class="flex items-center gap-2">
                                <input type="number" id="${userId}-${day}" class="input-day" value="${data.allowance_schedule[day] || 0}" step="15">
                                <span class="text-sm text-gray-400">min</span>
                            </div>
                        </div>`;
                });

                newHtml += `
                    <div class="kid-card p-6 ${isOvertime ? 'overtime-bg' : ''}">
                        <div class="flex justify-between items-start mb-6">
                            <h2 class="text-2xl font-black text-gray-800">${data.name || userId}</h2>
                            ${activeStationHtml}
                        </div>
                        <div class="text-center mb-4">
                            <div class="time-display ${isOvertime ? 'text-overtime' : ''}">${isOvertime ? '-' : ''}${formatTime(absTime)}</div>
                        </div>

                        <div class="mb-8">
                            ${barHtml}
                        </div>

                        <div class="mb-8">
                            <h3 class="text-sm font-bold text-gray-400 uppercase mb-4">Bonus Time</h3>
                            <div class="grid grid-cols-3 gap-3">
                                <button onclick="topUp('${userId}', 15)" class="btn-topup orange py-3">+15m</button>
                                <button onclick="topUp('${userId}', 30)" class="btn-topup orange py-3">+30m</button>
                                <button onclick="topUp('${userId}', 60)" class="btn-topup py-3">+1h</button>
                            </div>
                        </div>

                        <div class="bg-gray-50 rounded-2xl p-5 ${isOvertime ? '!bg-white/50' : ''}">
                            <div class="space-y-1 mb-5">${scheduleHtml}</div>
                            <button id="save-btn-${userId}" onclick="saveSchedule('${userId}')" class="w-full bg-gray-800 text-white font-bold py-4 rounded-xl">Save Schedule</button>
                        </div>
                    </div>`;
            }
            container.innerHTML = newHtml;
        }

        function renderSetup(ledger) {
            document.getElementById('penalty-input').value = ledger.system_rules?.overtime_multiplier || 1.0;

            const userContainer = document.getElementById('users-container');
            userContainer.innerHTML = '';
            for (const [userId, data] of Object.entries(ledger.users)) {
                userContainer.innerHTML += `
                    <div class="flex items-center gap-4 bg-gray-50 p-4 rounded-xl border border-gray-100">
                        <span class="font-bold text-gray-500 w-20 text-sm truncate">${userId}</span>
                        <input type="text" id="name-input-${userId}" value="${data.name || ''}" class="input-text !mb-0 flex-1">
                        <button id="name-btn-${userId}" onclick="saveUserName('${userId}')" class="bg-gray-200 hover:bg-gray-300 text-gray-700 font-bold py-2 px-4 rounded-lg">Rename</button>
                    </div>`;
            }

            const tokenContainer = document.getElementById('tokens-container');
            tokenContainer.innerHTML = '';
            for (const [mac, assignedUser] of Object.entries(ledger.tokens)) {
                let userOptions = '<option value="">-- Unassigned --</option>';
                for (const [uid, udata] of Object.entries(ledger.users)) {
                    userOptions += `<option value="${uid}" ${assignedUser === uid ? 'selected' : ''}>${udata.name || uid}</option>`;
                }
                tokenContainer.innerHTML += `
                    <div class="flex items-center gap-4 bg-gray-50 p-4 rounded-xl border border-gray-100 mb-2">
                        <span class="font-bold text-gray-500 w-36 text-sm font-mono truncate">${mac}</span>
                        <select onchange="assignToken('${mac}', this.value)" class="input-text !mb-0 flex-1 bg-white border-gray-300">${userOptions}</select>
                    </div>`;
            }

            const stContainer = document.getElementById('stations-container');
            stContainer.innerHTML = '';
            for (const [tagId, data] of Object.entries(ledger.stations)) {
                const b64Data = btoa(JSON.stringify(data));
                const isOverride = data.admin_override === true;
                const configuredDelay = data.integration?.delay_seconds || 0;

                let overrideHtml = '';
                if (data.type === 'webhook') {
                    overrideHtml = `
                        <button onclick="toggleOverride('${tagId}', ${isOverride})" class="flex items-center gap-2 mt-3 px-3 py-1.5 rounded-lg text-xs font-bold transition-colors ${isOverride ? 'bg-green-100 text-green-700 border border-green-300' : 'bg-gray-100 text-gray-500 border border-gray-200 hover:bg-gray-200'}">
                            <span class="relative flex h-2.5 w-2.5">
                              ${isOverride ? `<span class="animate-ping absolute inline-flex h-full w-full rounded-full bg-green-400 opacity-75"></span>` : ''}
                              <span class="relative inline-flex rounded-full h-2.5 w-2.5 ${isOverride ? 'bg-green-500' : 'bg-gray-400'}"></span>
                            </span>
                            Admin Override: ${isOverride ? 'ON (Unlocked)' : 'OFF'}
                        </button>`;
                }

                // Format multiple URLs for clean UI display if they exist
                const unlockUrls = (data.integration.unlock_url || '').split(',').map(u => u.trim()).filter(u => u);
                const lockUrls = (data.integration.lock_url || '').split(',').map(u => u.trim()).filter(u => u);

                let unlockDisplay = unlockUrls.map(u => `<div class="text-xs text-gray-400 truncate mb-1">🔓 ${u}</div>`).join('');
                let lockDisplay = lockUrls.map(u => `<div class="text-xs text-gray-400 truncate mb-1">🔒 ${u}</div>`).join('');

                let delayDisplay = configuredDelay > 0 ? `<div class="mt-2 text-xs font-bold text-gray-500 bg-gray-200 inline-block px-2 py-1 rounded">⏱ Delay between hooks: ${configuredDelay}s</div>` : '';

                stContainer.innerHTML += `
                    <div class="bg-gray-50 p-4 rounded-xl border border-gray-100 mb-4">
                        <div class="flex justify-between items-start mb-3">
                            <div>
                                <h3 class="font-bold text-lg leading-tight">${data.name}</h3>
                                <div class="flex items-center gap-2 mt-1">
                                    <span class="text-[10px] font-bold tracking-wider text-gray-500 uppercase bg-gray-200 px-2 py-0.5 rounded">${data.type}</span>
                                    <span class="text-xs font-mono text-gray-400">${tagId}</span>
                                </div>
                                ${overrideHtml}
                            </div>
                            <div class="flex gap-2">
                                <button onclick="editStation('${tagId}', '${b64Data}')" class="bg-gray-200 hover:bg-gray-300 text-gray-700 font-bold py-1 px-3 rounded-lg text-sm transition-colors">Edit</button>
                                <button id="del-btn-${tagId}" onclick="deleteStation('${tagId}')" class="bg-gray-200 hover:bg-red-500 hover:text-white text-gray-700 font-bold py-1 px-3 rounded-lg text-sm transition-colors">Delete</button>
                            </div>
                        </div>
                        ${data.type === 'webhook' ? `
                            ${unlockDisplay}
                            ${lockDisplay}
                            ${delayDisplay}
                        ` : '<div class="text-xs text-gray-400 italic">Manual Tracking (No Webhooks)</div>'}
                    </div>`;
            }
        }

        setInterval(fetchLedger, 3000);
        fetchLedger();
    </script>
</body>
</html>
"""

# ==========================================
# EXECUTION
# ==========================================
if __name__ == '__main__':
    print("[SYSTEM] Starting MQTT Background Thread...")
    threading.Thread(target=start_mqtt, daemon=True).start()

    print("[SYSTEM] Starting Flask Web Server on port 5000...")
    app.run(host="0.0.0.0", port=5000, debug=False)

[SYSTEM] Starting MQTT Background Thread...
[SYSTEM] Starting Flask Web Server on port 5000...
[MQTT] Failed to start: [Errno 111] Connection refused
 * Serving Flask app '__main__'
 * Debug mode: off


/tmp/ipykernel_1896/3780224140.py:164: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  mqtt_client = mqtt.Client(client_id="PiMasterLedger")
INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [15/Oct/2026 14:03:18] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Oct/2026 14:03:19] "GET /api/ledger HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Oct/2026 14:03:19] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [15/Oct/2026 14:03:22] "GET /api/ledger HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Oct/2026 14:03:25] "GET /api/ledger HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Oct/2026 14:03:27] "GET /api/ledger HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Oct/2026 14:03: